# 01 — Prompt-Boundary Alignment Predicts Useful Steering Layers

Reproduces paper **Fig. 2** (`gemma3_hum_concept_score_alignment.png`,
`gemma3_cs_align_coef3.0_scatter.png`), **Fig. 5**
(`gemma3_all_grid_concept_score_alignment.png`), **Fig. 6**
(`llama3_all_grid_concept_score_alignment.png`).

All inputs come from `results/statistics/` (per-layer 𝒜_c profiles) and
`results/steering_evaluations/` (the §3 grid-sweep summary CSVs).


In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import pearsonr, spearmanr

# Ensure working directory is the repo root so relative paths resolve correctly.
os.chdir(Path(__file__).resolve().parent.parent if '__file__' in dir() else Path.cwd().parent)

from grace.analysis.load_results import load_summary_results
from grace.diagnostics.alignment import alignment_per_layer

FIG_DIR = Path('Images')
FIG_DIR.mkdir(exist_ok=True)
MODELS = {'gemma3': 'google/gemma-3-27b-it', 'llama3': 'meta-llama/Llama-3.3-70B-Instruct'}
MODEL_SHORT = {'gemma3': 'Gemma3-27B', 'llama3': 'Llama-70B'}
CONCEPTS = sorted(p.stem for p in Path('concepts/gpt-5/extract').glob('*.json'))

# ── Plotting style ──────────────────────────────────────────────────────────
sns.set_style("whitegrid")
plt.rcParams.update({
    "figure.dpi": 200,
    "savefig.dpi": 300,
    "font.size": 14,
    "font.weight": "bold",
    "axes.labelweight": "bold",
    "axes.titleweight": "bold",
    "figure.titleweight": "bold",
})

# Per-concept colour / marker for scatter plots
_tab20 = plt.cm.tab20(np.linspace(0, 1, 20))
_markers = ["o", "s", "^", "v", "D", "P", "X", "*", "h", "p"] * 2
CONCEPT_STYLE = {
    c: {"color": _tab20[i], "marker": _markers[i]}
    for i, c in enumerate(CONCEPTS)
}
CONCEPT_DISPLAY = {c: c.replace('_', ' ').title() for c in CONCEPTS}

# Coefficient colours for dual-axis line charts
COEF_COLORS = ["red", "blue", "green"]

## Fig. 2a — Alignment vs. concept score for one example concept (humorous, gemma3)


In [ ]:
concept = 'humorous'
model_tag = 'gemma3'
model_name = MODELS[model_tag]
alignment = alignment_per_layer(model_name, concept, 'prompt_last')
rows = load_summary_results(model_name, concept, method='pv')
df = pd.DataFrame(rows).sort_values('layer')
coefs = sorted(df['coef'].unique())

fig, ax1 = plt.subplots(figsize=(12, 6))

# Left axis: concept_score lines per coefficient
for i, coef in enumerate(coefs):
    sub = df[df['coef'] == coef].sort_values('layer')
    ax1.plot(sub.layer, sub.mean_concept_score, 'o-',
             color=COEF_COLORS[i % len(COEF_COLORS)], markersize=7,
             linewidth=2.4, label=f'coef={coef}')

ax1.set_xlabel('Layer', fontsize=15, fontweight='bold')
ax1.set_ylabel('Concept Score', fontsize=17, fontweight='bold', color='#333')
ax1.set_ylim(0, 105)
ax1.tick_params(axis='y', labelcolor='#333', labelsize=12)
ax1.tick_params(axis='x', labelsize=12)
leg1 = ax1.legend(loc='upper left', fontsize=13, title='Concept Score',
                  title_fontproperties={'weight': 'bold', 'size': 12})

# Right axis: alignment
ax2 = ax1.twinx()
layers = sorted(alignment)
ax2.plot(layers, [alignment[l] for l in layers], 's-', color='black',
         markersize=8, linewidth=3, label=r'Alignment ($\mathcal{A}_c(\ell)$)', zorder=5)
ax2.set_ylabel('Prompt-Boundary Alignment', fontsize=17, fontweight='bold',
               color='black', labelpad=10)
ax2.tick_params(axis='y', labelcolor='black', labelsize=12)
ax2.set_ylim(0, 1.05)
ax2.grid(False, which='both', axis='both')
leg2 = ax2.legend(loc='upper right', fontsize=13)

ax1.set_title(f"{MODEL_SHORT[model_tag]} — {concept.replace('_', ' ').title()}",
              fontsize=18, fontweight='bold')
ax1.grid(True, alpha=0.5, axis='y')
ax1.tick_params(axis='x', grid_alpha=0)
for ax in [ax1, ax2]:
    for label in ax.get_xticklabels() + ax.get_yticklabels():
        label.set_fontweight('bold')

plt.tight_layout()
fig.savefig(FIG_DIR / 'gemma3_hum_concept_score_alignment.png',
            dpi=300, bbox_inches='tight')
plt.show()

## Fig. 2b — Pooled scatter: alignment vs. concept score across all concepts


In [ ]:
model_tag = 'gemma3'
model_name = MODELS[model_tag]

fig, ax = plt.subplots(figsize=(10, 7))
x_all, y_all = [], []

for concept in CONCEPTS:
    try:
        a = alignment_per_layer(model_name, concept, 'prompt_last')
        rows = [r for r in load_summary_results(model_name, concept, method='pv')
                if r['coef'] == 3.0]
    except FileNotFoundError:
        continue
    for r in rows:
        if r['layer'] in a:
            style = CONCEPT_STYLE[concept]
            ax.scatter(a[r['layer']], r['mean_concept_score'],
                       c=[style['color']], marker=style['marker'],
                       s=40, alpha=0.7, edgecolors='none',
                       label=CONCEPT_DISPLAY[concept]
                       if concept == CONCEPTS[0] or r is rows[0] else '')
            x_all.append(a[r['layer']])
            y_all.append(r['mean_concept_score'])

# Remove duplicate legend entries
handles, labels = ax.get_legend_handles_labels()
seen = set()
unique_h, unique_l = [], []
for h, l in zip(handles, labels):
    if l and l not in seen:
        seen.add(l)
        unique_h.append(h)
        unique_l.append(l)

x_all, y_all = np.array(x_all), np.array(y_all)
if len(x_all) >= 3:
    pr, pp = pearsonr(x_all, y_all)
    sr, sp = spearmanr(x_all, y_all)
    z = np.polyfit(x_all, y_all, 1)
    x_range = np.linspace(0, 1, 100)
    ax.plot(x_range, np.polyval(z, x_range), 'k--', alpha=0.5, linewidth=1.5)
    ax.text(0.05, 0.95,
            f"Pearson r = {pr:.3f} (p = {pp:.1e})\n"
            f"Spearman \u03c1 = {sr:.3f} (p = {sp:.1e})",
            transform=ax.transAxes, fontsize=13, fontweight='bold',
            va='top', ha='left',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8))

ax.set_xlim(0, 1.0)
ax.set_ylim(-2, 102)
ax.set_yticks(range(0, 101, 20))
ax.set_xlabel('Prompt-Boundary Alignment', fontsize=16, fontweight='bold')
ax.set_ylabel('Concept Score', fontsize=18, fontweight='bold')
ax.tick_params(axis='both', labelsize=13)
ax.set_title(f"{MODEL_SHORT[model_tag]} (coef=3.0)", fontsize=19, fontweight='bold')

# Concept legend on right
ax.legend(unique_h, unique_l, loc='center left', bbox_to_anchor=(1.01, 0.5),
          fontsize=10, ncol=1, markerscale=1.2)

plt.tight_layout()
fig.savefig(FIG_DIR / 'gemma3_cs_align_coef3.0_scatter.png',
            dpi=300, bbox_inches='tight')
plt.show()

## Fig. 5/6 — Per-concept alignment vs. concept-score grids (Gemma3 + Llama3)

5×4 small-multiples plot, one panel per concept. Same dual-axis layout as
Fig. 2a but rendered for every concept on each model.


In [ ]:
def grid_plot(model_name, model_tag, save_path, n_rows=5, n_cols=4):
    """5x4 grid of dual-axis line charts per concept, styled like Fig. 2."""
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(28, 5.5 * n_rows))
    fig.suptitle(f"{MODEL_SHORT[model_tag]} — Layer Profiles",
                 fontsize=20, fontweight='bold', y=1.01)

    for idx, concept in enumerate(CONCEPTS):
        row, col = divmod(idx, n_cols)
        ax = axes[row, col]
        try:
            a = alignment_per_layer(model_name, concept, 'prompt_last')
            all_rows = load_summary_results(model_name, concept, method='pv')
        except FileNotFoundError:
            ax.set_visible(False)
            continue
        if not all_rows:
            ax.set_visible(False)
            continue

        cdf = pd.DataFrame(all_rows).sort_values('layer')
        coefs = sorted(cdf['coef'].unique())

        # Compute correlation (best coef per layer)
        best_coef_df = cdf.loc[cdf.groupby('layer')['mean_concept_score'].idxmax()]
        corr_x, corr_y = [], []
        for _, r in best_coef_df.iterrows():
            if r['layer'] in a:
                corr_x.append(a[r['layer']])
                corr_y.append(r['mean_concept_score'])
        corr_x, corr_y = np.array(corr_x), np.array(corr_y)

        if len(corr_x) >= 3:
            pr, pp = pearsonr(corr_x, corr_y)
            sr, sp = spearmanr(corr_x, corr_y)
            corr_str = (f"Pearson r={pr:.2f} (p={pp:.1e})  "
                        f"Spearman \u03c1={sr:.2f} (p={sp:.1e})")
        else:
            corr_str = ""

        # Left axis: concept_score lines per coef
        cmap_vals = plt.cm.viridis(np.linspace(0.15, 0.85, len(coefs)))
        for i, coef in enumerate(coefs):
            sub = cdf[cdf['coef'] == coef].sort_values('layer')
            ax.plot(sub['layer'], sub['mean_concept_score'], 'o-',
                    color=cmap_vals[i], markersize=4, linewidth=1.5,
                    label=f'coef={coef}')
        ax.set_ylabel('Concept Score', color='#333', fontsize=12, fontweight='bold')
        ax.tick_params(axis='y', labelcolor='#333', labelsize=10)
        ax.set_ylim(0, 105)

        # Right axis: alignment
        ax2 = ax.twinx()
        color_orange = '#e67e22'
        layers_sorted = sorted(a)
        alignment_vals = [a[l] for l in layers_sorted]
        ax2.plot(layers_sorted, alignment_vals, 's--', color=color_orange,
                 markersize=4, linewidth=1.5)
        ax2.set_ylabel(r'$\mathcal{A}_c(\ell)$', color=color_orange,
                       fontsize=12, fontweight='bold')
        ax2.tick_params(axis='y', labelcolor=color_orange, labelsize=10)
        ax2.set_ylim(0, 1)

        ax.set_title(f"{concept}\n{corr_str}",
                     fontsize=11, fontweight='bold')
        ax.set_xlabel('Layer', fontsize=12, fontweight='bold')
        ax.tick_params(axis='x', labelsize=10)

    # Shared legend from first subplot
    handles, labels = axes[0, 0].get_legend_handles_labels()
    fig.legend(handles, labels, loc='lower center', ncol=len(handles),
               fontsize=12, bbox_to_anchor=(0.5, -0.01),
               prop={'weight': 'bold'})

    for idx in range(len(CONCEPTS), n_rows * n_cols):
        row, col = divmod(idx, n_cols)
        axes[row, col].set_visible(False)

    plt.tight_layout()
    fig.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()

grid_plot(MODELS['gemma3'], 'gemma3',
          FIG_DIR / 'gemma3_all_grid_concept_score_alignment.png')
grid_plot(MODELS['llama3'], 'llama3',
          FIG_DIR / 'llama3_all_grid_concept_score_alignment.png')